In [39]:
#================PACKAGES==============#
import torch
import torchvision
from torch import nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader


#================DISCRIMINATOR=========#
# a discriminator-we go from img generated to feature maps again i.e mirror of generator
class Discriminator(nn.Module):
    def __init__(self,img_dim):
        super().__init__()
        self.stack = nn.Sequential(
            nn.Linear(img_dim, 128),
            nn.LeakyReLU(0.01),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.stack(x)



#================GENERATOR=============#
class Generator(nn.Module):
    def __init__(self,z_dim, img_dim):
        super().__init__()
        self.stack = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.01),
            nn.Linear(256, img_dim),
            nn.Tanh()
        )
    def forward(self,x):
        return self.stack(x)



#================HYPERPARAMETERS========#
device = "cuda" if torch.cuda.is_available() else "cpu"
lr = 3e-4
z_dim = 64
img_dim = 28 * 28 * 1  # 784
batch_size = 32
num_epochs = 1






#================INSTANTIATE THE MODELS AND NOISE===================#

"""
instantiate the model, loss function, optimizer, transforms, and SummaryWriter
"""
disc = Discriminator(img_dim).to(device)
gen = Generator(z_dim, img_dim).to(device)
fixed_noise = torch.randn((batch_size, z_dim))
loss_fn = nn.BCELoss()
optim_disc = torch.optim.Adam(disc.parameters(), lr = lr)
optim_gen = torch.optim.Adam(gen.parameters(), lr = lr)
transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5)),
])
writer_fake = SummaryWriter(f'logs/fake')
writer_real = SummaryWriter(f'logs/real')

dataset = datasets.MNIST(root = 'data',download = False, transform = transforms)

loader = DataLoader(
    dataset,
    batch_size,
)



#==================TRAINING PROCESS=================================#

"""
add images to the tensorboard to show the step process
"""
from tqdm.auto import tqdm
step = 0
for epoch in tqdm(range(num_epochs)):
    for batch_idx, (data,_) in enumerate(loader):
        #we have to calculate the loss for the discriminator and generator
        real = data.view(-1, 784)
        batch_size = real.shape[0]
        noise = torch.randn((batch_size, z_dim))
        fake = gen(noise)
        disc_real = disc(real).view(-1)
        disc_fake = disc(fake).view(-1)
        lossD = loss_fn(disc_real, torch.ones_like(disc_real))
        lossF = loss_fn(disc_fake, torch.zeros_like(disc_real))
        loss = (lossD + lossF)/2
        optim_disc.zero_grad()
        loss.backward(retain_graph = True)
        optim_disc.step()

        output = disc(fake).view(-1)
        lossG = loss_fn(output, torch.ones_like(output))
        optim_gen.zero_grad()
        lossG.backward(retain_graph = True)
        optim_gen.step()

        if batch_idx == 0:
            print(
                f"Epoch [{epoch}/{num_epochs}] Batch {batch_idx}/{len(loader)} \
                      Loss D: {loss:.4f}, loss G: {lossG:.4f}"
            )
    
            with torch.no_grad():
                fake = gen(fixed_noise).reshape(-1, 1, 28, 28)
                real_data = real.reshape(-1, 1, 28, 28)
                img_grid_fake = torchvision.utils.make_grid(fake, normalize = True)
                img_grid_real = torchvision.utils.make_grid(real_data, normalize = True)
    
                writer_fake.add_image('Mnist fake image',img_grid_fake, global_step = step)
                writer_real.add_image('Mnist real image',img_grid_real, global_step = step)
                
                step +=1

  0%|          | 0/1 [00:00<?, ?it/s]

Epoch [0/1] Batch 0/1875                       Loss D: 0.7697, loss G: 0.7113
torch.Size([32, 1, 28, 28])


In [41]:
%load_ext tensorboard
%tensorboard --logdir logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 22468), started 0:25:50 ago. (Use '!kill 22468' to kill it.)

In [47]:
!pip install ipywidgets

In [49]:
from tqdm.auto import tqdm
import time

num_test = 10
for i in tqdm(range(num_test)):
    tqdm.write(f"Processing item: {i}")
    time.sleep(0.1) # Added sleep just to make the animation visible


  0%|          | 0/10 [00:00<?, ?it/s]

Processing item: 0
Processing item: 1
Processing item: 2
Processing item: 3
Processing item: 4
Processing item: 5
Processing item: 6
Processing item: 7
Processing item: 8
Processing item: 9
